# Ch01-02: 결측값 메커니즘과 다중 대치법


## 학습 목표

- MCAR, MAR, MNAR의 수학적 정의와 검정법을 이해한다
- Little's MCAR 검정을 구현하고 해석한다
- MICE 알고리즘의 수리적 기반과 Rubin's rule을 이해한다
- KNN 대치법의 거리 가중 방식을 구현하고 성능을 비교한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 결측 메커니즘 (Rubin, 1976)

완전 데이터 $Y=(Y_{\text{obs}},Y_{\text{mis}})$, 결측 지시 행렬 $R$:

- **MCAR**: $P(R|Y_{\text{obs}},Y_{\text{mis}},\psi)=P(R|\psi)$
- **MAR**: $P(R|Y_{\text{obs}},Y_{\text{mis}},\psi)=P(R|Y_{\text{obs}},\psi)$
- **MNAR**: 위의 조건 불만족

### Little's MCAR 검정

$$d^2 = \sum_{j=1}^J n_j(\bar{y}_j-\hat{\mu}_j)^T\hat{\Sigma}_j^{-1}(\bar{y}_j-\hat{\mu}_j) \sim \chi^2\left(\sum_j p_j - p\right)$$

### MICE (Multiple Imputation by Chained Equations)

각 변수를 나머지의 조건부 모형으로 순환 대치. $m$개 대치 데이터에 Rubin's rule:

$$\bar{Q}=\frac{1}{m}\sum\hat{Q}_l, \quad T=\bar{U}+(1+\frac{1}{m})B$$

### KNN 대치

$$\hat{y}_{ij} = \frac{\sum_{l\in N_k(i)} w_{il}\cdot y_{lj}}{\sum_{l\in N_k(i)} w_{il}}, \quad w_{il}=\frac{1}{d(i,l)^2}$$


## 구현 가이드

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import seaborn as sns

np.random.seed(42)
n = 500
X1 = np.random.normal(50, 10, n)
X2 = 0.6*X1 + np.random.normal(0, 8, n)
X3 = np.random.normal(30, 5, n)
df_full = pd.DataFrame({'X1':X1, 'X2':X2, 'X3':X3})

# MCAR
df_mcar = df_full.copy()
df_mcar[np.random.random((n,3))<0.2] = np.nan

# MAR (X1 클수록 X2 결측)
df_mar = df_full.copy()
prob = 1/(1+np.exp(-(X1-55)/5))
df_mar.loc[np.random.random(n)<prob, 'X2'] = np.nan

# 결측 패턴 시각화
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(df_mcar.isnull(), cbar=False, ax=axes[0], yticklabels=False)
axes[0].set_title('MCAR'); sns.heatmap(df_mar.isnull(), cbar=False, ax=axes[1], yticklabels=False)
axes[1].set_title('MAR'); plt.tight_layout(); plt.show()

# MICE / KNN 대치
df_mice = pd.DataFrame(IterativeImputer(max_iter=20, random_state=42).fit_transform(df_mcar), columns=df_mcar.columns)
df_knn = pd.DataFrame(KNNImputer(n_neighbors=5, weights='distance').fit_transform(df_mcar), columns=df_mcar.columns)


---
## 연습 문제

### 문제 1 [★]

5변수 데이터에 MCAR/MAR/MNAR 결측 20%씩 생성하고, 결측 전후 분포/상관행렬 변화를 시각화하여 어떤 메커니즘이 편향을 유발하는지 분석하라.

In [ ]:
# TODO: 5변수 상관 데이터
# TODO: 3종 결측 생성
# TODO: 분포/상관행렬 비교


### 문제 2 [★★]

Little's MCAR 검정을 직접 구현하라 (외부 패키지 금지). EM으로 전체 mu/Sigma 추정, 카이제곱 통계량 계산. MCAR vs MAR 데이터에서 검정력 비교.

> **힌트**: EM의 E-step에서 조건부 기대값/공분산을 갱신한다.

In [ ]:
def littles_mcar_test(df):
    # TODO
    pass


### 문제 3 [★★]

MICE를 직접 구현하라 (sklearn 금지). Bayesian linear regression 사후 예측 분포에서 샘플링. m=5 대치 데이터에 Rubin's rule로 결합 추정치와 CI를 구하라.

In [ ]:
def mice_impute(df, m=5, max_iter=20):
    # TODO
    pass

def rubins_rule(estimates, variances):
    # TODO
    pass


### 문제 4 [★★★]

MNAR 민감도 분석 수행.
1) Y=b0+b1*X1+b2*X2+e 적합
2) MNAR로 Y 30% 결측
3) 틸팅 모형 logit(P(R=0))=a0+a1*Y에서 a1을 변화시키며 b1 편향 분석
4) 추정치 변화 그래프와 해석

In [ ]:
# TODO: 완전 데이터 + 참 회귀계수
# TODO: 다양한 a1에서 대치 후 분석
# TODO: 편향 그래프


---
## 참고 자료

- Rubin, D.B. (1976). Inference and Missing Data. Biometrika.
- Little, R.J.A. (1988). A Test of MCAR. JASA.
- van Buuren, S. (2018). Flexible Imputation of Missing Data. CRC Press.
